# D2.5 · Replay and forensics

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.4 · Containment at machine speed](https://spbreed.github.io/cyber-commons/lessons/D2.4.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Replay an agent run for a regulator-grade record.

**Why a security engineer needs it.** Non-determinism as an evidentiary problem. The control it builds is: log at design time what replay will need.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Forensics on a non-deterministic actor asks a question classical forensics never had to: not just what it did, but what it saw and what it decided. If the context was not recorded, the decision cannot be reconstructed at all.

> **At CyberTravels.** Not just what the agent did, but what it saw and what it decided. If the booking note that triggered the refund was not recorded, the decision cannot be reconstructed at all. R11.

## 2 · The framework

```
   classical forensics        agentic forensics
   +------------------+       +----------------------------+
   | what did it do   |       | what did it do             |
   |                  |       | what did it SEE            |
   |                  |       | what did it DECIDE, and why|
   +------------------+       +----------------------------+

   if the context was not recorded, the decision cannot be reconstructed
```

Forensics for an agent means answering: *why did it do that?*

For ordinary software the answer is in the code. For an agent the answer is in
the run — the prompts, the tool results it saw, the model version, the sampling.
Reproduce those four and the run is deterministic. Miss one and you can describe
what happened but never demonstrate it, which matters the moment anyone
disputes your conclusion.

The field teams miss most often is the **model version**, and it is the one that
silently invalidates everything else: a provider-side upgrade changes the
behaviour with no change on your side, so a reconstruction performed after the
upgrade does not reproduce the incident that happened before it.

## 3 · The procedure, as a skill

Replay needs five inputs and the typical production run records three. The skill checks each against a real record, then replays under two later model versions — where a different action means the original decision cannot be reproduced at all.

### The skill — [`skills/response/run-replayability-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/run-replayability-audit/SKILL.md)

```yaml
name: run-replayability-audit
description: >-
  Check whether an incident run can be replayed at all — model version, seed,
  prompt, tool results, retrieved context — and what a later model version does
  to the replay. Use when forensics needs to know why the agent did what it did.
allowed-tools: Read, Grep, Glob
```

# Replay needs five things and production records three

"Why did the agent do that" is answerable only if the run can be re-run under the
conditions it ran in. That needs the model version, the sampling parameters and
seed, the exact prompt, every tool result and the retrieved context. A typical
production run records enough to see what happened and not enough to reproduce
it.

## When to use this

Before an incident, as a readiness check, and during one, to establish honestly
whether the reconstruction is possible.

## Procedure

**1 — List the five inputs and check each against a real run record.** Model
version and seed are the two usually missing, and their absence is decisive
rather than inconvenient.

**2 — Attempt the replay.** If any input is missing, say what the replay can and
cannot establish. A partial replay is still useful for the tool path and useless
for the reasoning.

**3 — Replay under later model versions.** The provider has probably upgraded.
Record whether the action changes: if it does, the original decision cannot be
reproduced at all, and that is a finding about the estate rather than about the
incident.

**4 — Cost full instrumentation.** Storage and latency for recording everything,
against the incidents where you needed it. Present both; the answer is usually to
instrument the high-tier agents only, and that is a defensible decision when the
numbers are attached.

**5 — Record what the estate has chosen.** Which agents are replayable and which
are not, so nobody assumes during an incident.

## Output contract

```json
{
  "inputs": [{"name": "str", "recorded": false}],
  "replay": {"possible": false, "establishes": ["str"], "cannot_establish": ["str"]},
  "version_drift": [{"version": "str", "action": "str", "same_as_original": false}],
  "cost": {"storage_per_run": "str", "latency_ms": 0},
  "policy": [{"tier": "str", "fully_instrumented": true}]
}
```

## Failure modes

- **Assuming replay is possible.** Check the record before promising it.
- **Replaying on the current model.** It is not the one that acted.
- **Instrumenting everything or nothing.** Tier it and write the choice down.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/run-replayability-audit/scripts/run_replayability_audit.py
SCRIPT = "skills/response/run-replayability-audit/scripts/run_replayability_audit.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Only the fully instrumented run is replayable; the typical production run is missing the model version and seed. Replaying the incident under two later model versions produces a different action, so the original run does not reproduce. Adding the two cheapest fields (model version and seed) makes the typical production run replayable.

## Your turn

Add model version and seed to your agent's run records this week. Both are one field each, and together they are the difference between forensics and storytelling.

---

**Next → [D2.6 · Post-incident change surface](https://spbreed.github.io/cyber-commons/lessons/D2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*